# HW4: Temperature Forecast Web App with CWA API

使用中央氣象署（CWA）開放資料 API 取得台灣六大區域一週的氣溫預報，
分析 JSON 找出每日最高／最低氣溫，存進 SQLite 資料庫，
最後用 Streamlit 做成互動式氣溫預報 Web App。

## ⚠️ 兩件開始前要先知道的事

### 1. 授權碼（API key）
本作業**必須使用自己申請的授權碼**，沿用課程範例的金鑰該題以 0 分計算。

申請方式：到 [氣象資料開放平臺](https://opendata.cwa.gov.tw/) 註冊會員 →
「會員資訊 / 取得授權碼」→ 複製格式類似 `CWA-XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX` 的字串。

下面的程式會優先讀取環境變數 `CWA_API_KEY`，讀不到才用 `MY_API_KEY` 變數，
所以金鑰不會被寫死在程式碼裡。

### 2. 資料集異動：`F-A0010-001` 已下架
作業說明使用的「一週農業氣象預報」`F-A0010-001`，
目前在氣象資料開放平臺上已查不到，呼叫 API 會回傳 `404 Resouce not found`。

因此本作業改用同樣是 CWA 官方、同樣是 JSON 格式、同樣涵蓋未來一週的
**`F-C0032-005`「一般天氣預報－一週縣市天氣預報」**，
再依氣象署的分區方式把 22 個縣市彙整成作業要求的六大區域：

| 區域 | 涵蓋縣市 |
|------|----------|
| 北部地區 | 臺北市、新北市、基隆市、桃園市、新竹縣、新竹市 |
| 中部地區 | 苗栗縣、臺中市、彰化縣、南投縣、雲林縣 |
| 南部地區 | 嘉義縣、嘉義市、臺南市、高雄市、屏東縣 |
| 東北部地區 | 宜蘭縣 |
| 東部地區 | 花蓮縣 |
| 東南部地區 | 臺東縣 |

程式仍會**先嘗試原本的 `F-A0010-001`**，抓不到才自動改用 `F-C0032-005`，
所以未來原資料集若恢復上架，這份程式不用改就能繼續使用。

## HW4-1: Fetch Weather Forecast Data
- 使用 CWA API 取得台灣北部、中部、南部、東北部、東部及東南部地區一週的天氣預報資料（JSON 格式）
- 使用 Requests 套件調用 API
- 使用 `json.dumps` 觀察獲得的資料

### Step 1: Install the dependencies

In [1]:
%pip install -q requests pandas streamlit plotly

Note: you may need to restart the kernel to use updated packages.


### Step 2: 設定授權碼與 API 位址

> 在 terminal 先執行 `export CWA_API_KEY="你的授權碼"`，
> 或直接把下面的 `MY_API_KEY` 改成你的授權碼。

In [2]:
import json
import os

import requests

# 👉 把這裡換成「你自己」申請的授權碼（或改用環境變數 CWA_API_KEY）
MY_API_KEY = "CWA-XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"

API_KEY = os.getenv("CWA_API_KEY", "").strip() or MY_API_KEY

if not API_KEY or API_KEY.startswith("CWA-XXXX"):
    raise SystemExit("請先設定你自己的 CWA 授權碼！")

# CWA 檔案型開放資料的共同進入點
BASE_URL = "https://opendata.cwa.gov.tw/fileapi/v1/opendataapi"

# 作業原本指定的資料集；目前已下架，保留作為第一順位嘗試
PRIMARY_DATASET = "F-A0010-001"   # 一週農業氣象預報
# 替代資料集：一樣是 CWA 官方、JSON、未來一週的預報
FALLBACK_DATASET = "F-C0032-005"  # 一般天氣預報－一週縣市天氣預報

# 作業要求的六大區域
REGIONS = ["北部地區", "中部地區", "南部地區", "東北部地區", "東部地區", "東南部地區"]

print(f"授權碼已設定（共 {len(API_KEY)} 碼）")  # 不印出內容，避免金鑰跟著作業一起交出去

授權碼已設定（共 40 碼）


/Users/albertchen/Desktop/hw4_weather_forecast/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Step 3: 使用 Requests 呼叫 CWA API

In [3]:
def fetch_dataset(dataset_id: str) -> dict:
    """呼叫 CWA API 取得指定資料集的 JSON。

    授權碼透過 params 傳送，不直接串在網址字串裡，
    這樣 print 出來時比較不會不小心把金鑰外洩。
    """
    response = requests.get(
        f"{BASE_URL}/{dataset_id}",
        params={"Authorization": API_KEY, "downloadType": "WEB", "format": "JSON"},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


# 先試作業指定的資料集，失敗就改用替代資料集
try:
    dataset_id = PRIMARY_DATASET
    data = fetch_dataset(dataset_id)
    print(f"✅ 成功取得 {dataset_id}（一週農業氣象預報）")
except requests.HTTPError as error:
    print(f"⚠️  {PRIMARY_DATASET} 無法取得（{error.response.status_code}），改用替代資料集")
    dataset_id = FALLBACK_DATASET
    data = fetch_dataset(dataset_id)
    print(f"✅ 成功取得 {dataset_id}（一週縣市天氣預報）")

⚠️  F-A0010-001 無法取得（404），改用替代資料集


✅ 成功取得 F-C0032-005（一週縣市天氣預報）


### Step 4: 使用 `json.dumps` 觀察獲得的資料

In [4]:
# 原始 JSON 很長，先只看最外層結構
print(json.dumps(data, indent=4, ensure_ascii=False)[:1500])
print("\n...（以下省略）")

{
    "cwaopendata": {
        "@xmlns": "urn:cwa:gov:tw:cwacommon:0.1",
        "identifier": "b49f4eb7-1fc2-d1be-9497-b379651556c8",
        "sender": "weather@cwa.gov.tw",
        "sent": "2026-09-16T16:39:13+08:00",
        "status": "Actual",
        "msgType": "Issue",
        "source": "MFC",
        "dataid": "C0032-005",
        "scope": "Public",
        "dataset": {
            "datasetInfo": {
                "datasetDescription": "一週縣市天氣預報",
                "issueTime": "2026-09-16T17:00:00+08:00",
                "update": "2026-09-16T16:39:13+08:00"
            },
            "location": [
                {
                    "locationName": "臺北市",
                    "weatherElement": [
                        {
                            "elementName": "Wx",
                            "time": [
                                {
                                    "startTime": "2026-09-16T18:00:00+08:00",
                                    "endTime": "2026-09-17T06:

In [5]:
# 看一下這份資料的基本資訊
root = data["cwaopendata"]
info = root["dataset"]["datasetInfo"] if dataset_id == FALLBACK_DATASET else root

print("資料集代號 :", root.get("dataid"))
print("發布時間   :", root.get("sent"))
print("資料說明   :", info.get("datasetDescription", root.get("datasetName")))
print("最外層欄位 :", list(root.keys()))

資料集代號 : C0032-005
發布時間   : 2026-09-16T16:39:13+08:00
資料說明   : 一週縣市天氣預報
最外層欄位 : ['@xmlns', 'identifier', 'sender', 'sent', 'status', 'msgType', 'source', 'dataid', 'scope', 'dataset']


## HW4-2: Analyze the Fetched Data
- 分析 JSON 結構，找出每日 `MaxT`（最高溫）與 `MinT`（最低溫）的位置
- 注意：作業中的 `Region` 在資料集裡叫做 `Location`
- 使用 `json.dumps` 觀察提取出來的資料

### Step 1: 分析資料結構，找出氣溫藏在哪裡

In [6]:
def show_structure(node, path="cwaopendata", depth=0, max_depth=4):
    """把 JSON 一層一層印出來，方便找出氣溫資料的路徑。"""
    if depth > max_depth:
        return
    pad = "    " * depth
    if isinstance(node, dict):
        for key, value in node.items():
            kind = type(value).__name__
            size = f"(共 {len(value)} 筆)" if isinstance(value, (list, dict)) else f"= {str(value)[:40]}"
            print(f"{pad}{key} : {kind} {size}")
            show_structure(value, f"{path}/{key}", depth + 1, max_depth)
    elif isinstance(node, list) and node:
        print(f"{pad}[0] ← 以第 1 筆為例（共 {len(node)} 筆）")
        show_structure(node[0], f"{path}[0]", depth + 1, max_depth)


show_structure(data)

cwaopendata : dict (共 10 筆)
    @xmlns : str = urn:cwa:gov:tw:cwacommon:0.1
    identifier : str = b49f4eb7-1fc2-d1be-9497-b379651556c8
    sender : str = weather@cwa.gov.tw
    sent : str = 2026-09-16T16:39:13+08:00
    status : str = Actual
    msgType : str = Issue
    source : str = MFC
    dataid : str = C0032-005
    scope : str = Public
    dataset : dict (共 2 筆)
        datasetInfo : dict (共 3 筆)
            datasetDescription : str = 一週縣市天氣預報
            issueTime : str = 2026-09-16T17:00:00+08:00
            update : str = 2026-09-16T16:39:13+08:00
        location : list (共 22 筆)
            [0] ← 以第 1 筆為例（共 22 筆）
                locationName : str = 臺北市
                weatherElement : list (共 3 筆)


In [7]:
# 結論：氣溫資料的位置
#
# F-C0032-005（一週縣市天氣預報）
#   cwaopendata / dataset / location[] / weatherElement[] / time[] / parameter / parameterName
#   其中 weatherElement 的 elementName 有 Wx（天氣現象）、MaxT（最高溫）、MinT（最低溫）
#
# F-A0010-001（一週農業氣象預報，已下架）
#   cwaopendata / resources / resource / data / agrWeatherForecasts
#             / weatherForecasts / location[] / weatherElements / MaxT / daily[]

locations = data["cwaopendata"]["dataset"]["location"]
print("資料中的 Location（地區）共", len(locations), "個：")
print([loc["locationName"] for loc in locations])
print()
print("每個 Location 底下的 weatherElement：")
print([element["elementName"] for element in locations[0]["weatherElement"]])
print()
print("MaxT 的第 1 筆時間區間長這樣：")
maxt = [e for e in locations[0]["weatherElement"] if e["elementName"] == "MaxT"][0]
print(json.dumps(maxt["time"][0], indent=4, ensure_ascii=False))

資料中的 Location（地區）共 22 個：
['臺北市', '新北市', '桃園市', '臺中市', '臺南市', '高雄市', '基隆市', '新竹縣', '新竹市', '苗栗縣', '彰化縣', '南投縣', '雲林縣', '嘉義縣', '嘉義市', '屏東縣', '宜蘭縣', '花蓮縣', '臺東縣', '澎湖縣', '金門縣', '連江縣']

每個 Location 底下的 weatherElement：
['Wx', 'MaxT', 'MinT']

MaxT 的第 1 筆時間區間長這樣：
{
    "startTime": "2026-09-16T18:00:00+08:00",
    "endTime": "2026-09-17T06:00:00+08:00",
    "parameter": {
        "parameterName": "25",
        "parameterUnit": "C"
    }
}


### Step 2: 只取出每個地區每日的 `MaxT` 與 `MinT`

`F-C0032-005` 是「縣市 × 每 12 小時（日／夜）」的資料，
要轉成作業要的「六大區域 × 每日」需要兩個步驟：

1. **縣市 → 區域**：依氣象署分區把縣市對應到六大區域
2. **日夜 → 每日**：同一天同一區域的所有時段裡，取最大值當 `MaxT`、最小值當 `MinT`

整理完的結果會做成和原本 `F-A0010-001` 一樣的格式，
這樣後面存資料庫的程式就跟作業說明完全一致。

In [8]:
from collections import defaultdict

# 氣象署的分區方式：縣市 → 六大區域（澎湖、金門、連江屬離島，不在六大區域內）
REGION_OF_COUNTY = {
    "臺北市": "北部地區", "新北市": "北部地區", "基隆市": "北部地區",
    "桃園市": "北部地區", "新竹縣": "北部地區", "新竹市": "北部地區",
    "苗栗縣": "中部地區", "臺中市": "中部地區", "彰化縣": "中部地區",
    "南投縣": "中部地區", "雲林縣": "中部地區",
    "嘉義縣": "南部地區", "嘉義市": "南部地區", "臺南市": "南部地區",
    "高雄市": "南部地區", "屏東縣": "南部地區",
    "宜蘭縣": "東北部地區",
    "花蓮縣": "東部地區",
    "臺東縣": "東南部地區",
}

DAYS = 7  # 一週


def extract_from_county_forecast(data: dict) -> list:
    """F-C0032-005：把縣市、日夜的預報彙整成「六大區域 × 每日」。"""
    # bucket[(區域, 日期)] = {"MaxT": [...], "MinT": [...]}
    bucket = defaultdict(lambda: {"MaxT": [], "MinT": []})

    for location in data["cwaopendata"]["dataset"]["location"]:
        region = REGION_OF_COUNTY.get(location["locationName"])
        if region is None:          # 離島縣市不列入六大區域
            continue
        for element in location["weatherElement"]:
            name = element["elementName"]
            if name not in ("MaxT", "MinT"):
                continue
            for slot in element["time"]:
                date = slot["startTime"][:10]           # "2026-09-17T06:00:00+08:00" → "2026-09-17"
                bucket[(region, date)][name].append(int(slot["parameter"]["parameterName"]))

    # 每個區域只保留最近的 7 天
    # 註：因為預報是每天下午更新，第一天（今天）通常只剩下傍晚之後的時段，
    #     所以今天的 MaxT 會比整日實際高溫略低，這是資料本身的特性。
    dates = sorted({date for _region, date in bucket})[:DAYS]

    forecasts = []
    for region in REGIONS:
        daily_max, daily_min = [], []
        for date in dates:
            values = bucket.get((region, date))
            if not values or not values["MaxT"] or not values["MinT"]:
                continue
            daily_max.append({"dataDate": date, "temperature": str(max(values["MaxT"]))})
            daily_min.append({"dataDate": date, "temperature": str(min(values["MinT"]))})
        forecasts.append({
            "locationName": region,
            "weatherElements": {"MaxT": {"daily": daily_max}, "MinT": {"daily": daily_min}},
        })
    return forecasts


def extract_from_agr_forecast(data: dict) -> list:
    """F-A0010-001：本來就是「區域 × 每日」，只要把不需要的 Wx 拿掉。"""
    forecasts = (
        data["cwaopendata"]["resources"]["resource"]["data"]
        ["agrWeatherForecasts"]["weatherForecasts"]["location"]
    )
    for region in forecasts:
        region["weatherElements"].pop("Wx", None)
    return forecasts


if dataset_id == PRIMARY_DATASET:
    temperature_forecasts = extract_from_agr_forecast(data)
else:
    temperature_forecasts = extract_from_county_forecast(data)

print(f"整理完成：{len(temperature_forecasts)} 個地區，"
      f"每區 {len(temperature_forecasts[0]['weatherElements']['MaxT']['daily'])} 天")

整理完成：6 個地區，每區 7 天


使用 `json.dumps` 觀察提取出來的資料：

In [9]:
# 完整印出第一個地區，確認格式正確
print(json.dumps(temperature_forecasts[0], indent=4, ensure_ascii=False))

{
    "locationName": "北部地區",
    "weatherElements": {
        "MaxT": {
            "daily": [
                {
                    "dataDate": "2026-09-16",
                    "temperature": "27"
                },
                {
                    "dataDate": "2026-09-17",
                    "temperature": "30"
                },
                {
                    "dataDate": "2026-09-18",
                    "temperature": "31"
                },
                {
                    "dataDate": "2026-09-19",
                    "temperature": "31"
                },
                {
                    "dataDate": "2026-09-20",
                    "temperature": "30"
                },
                {
                    "dataDate": "2026-09-21",
                    "temperature": "30"
                },
                {
                    "dataDate": "2026-09-22",
                    "temperature": "30"
                }
            ]
        },
        "MinT": {
 

In [10]:
# 六個地區都印出來（只印摘要，避免版面太長）
for region in temperature_forecasts:
    name = region["locationName"]
    maxt = [int(d["temperature"]) for d in region["weatherElements"]["MaxT"]["daily"]]
    mint = [int(d["temperature"]) for d in region["weatherElements"]["MinT"]["daily"]]
    print(f"{name:6}  最高溫 {maxt}  最低溫 {mint}")

北部地區    最高溫 [27, 30, 31, 31, 30, 30, 30]  最低溫 [23, 23, 24, 24, 24, 22, 22]
中部地區    最高溫 [29, 32, 33, 33, 32, 32, 32]  最低溫 [22, 22, 22, 23, 23, 21, 21]
南部地區    最高溫 [29, 33, 34, 34, 33, 32, 32]  最低溫 [24, 24, 25, 25, 25, 24, 24]
東北部地區   最高溫 [26, 29, 29, 30, 30, 30, 30]  最低溫 [23, 23, 24, 24, 24, 22, 22]
東部地區    最高溫 [27, 30, 30, 30, 30, 30, 30]  最低溫 [24, 24, 25, 25, 25, 24, 24]
東南部地區   最高溫 [28, 30, 31, 30, 31, 30, 30]  最低溫 [25, 25, 25, 25, 25, 25, 25]


## HW4-3: Save the Temperature Data to SQLite Database
- 建立 SQLite 資料庫 `data.db`
- 建立資料表 `TemperatureForecasts`，欄位包含：
  - `id`：主鍵
  - `regionName`：地區名稱
  - `dataDate`：預報日期
  - `MaxT`：最高氣溫
  - `MinT`：最低氣溫
- 存好之後查詢確認資料正確

### Step 1: 把氣溫資料存進 SQLite 資料庫

In [11]:
import sqlite3

DB_NAME = "data.db"
TABLE_NAME = "TemperatureForecasts"

conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# 重跑這份 notebook 時先清掉舊資料，避免重複累加
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")

cursor.execute(f"""
CREATE TABLE {TABLE_NAME} (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    regionName TEXT NOT NULL,
    dataDate   TEXT NOT NULL,
    MaxT       INTEGER,
    MinT       INTEGER
)
""")

# 逐區、逐日把資料寫進資料表
inserted = 0
for region in temperature_forecasts:
    region_name = region["locationName"]
    maxt_daily = region["weatherElements"]["MaxT"]["daily"]
    mint_daily = region["weatherElements"]["MinT"]["daily"]

    for maxt, mint in zip(maxt_daily, mint_daily):
        cursor.execute(
            f"INSERT INTO {TABLE_NAME} (regionName, dataDate, MaxT, MinT) VALUES (?, ?, ?, ?)",
            (region_name, maxt["dataDate"], int(maxt["temperature"]), int(mint["temperature"])),
        )
        inserted += 1

conn.commit()
conn.close()

print(f"已寫入 {inserted} 筆資料到 {DB_NAME} 的 {TABLE_NAME} 資料表")

已寫入 42 筆資料到 data.db 的 TemperatureForecasts 資料表


### Step 2: 查詢資料庫，確認資料正確存入

**查詢 1：列出所有地區名稱**

In [12]:
import pandas as pd

conn = sqlite3.connect(DB_NAME)

query = "SELECT DISTINCT regionName FROM TemperatureForecasts"
region_names = pd.read_sql_query(query, conn)

print(region_names)

conn.close()

  regionName
0       北部地區
1       中部地區
2       南部地區
3      東北部地區
4       東部地區
5      東南部地區


**查詢 2：列出 `中部地區` 的氣溫資料**

In [13]:
conn = sqlite3.connect(DB_NAME)

selected_region = "中部地區"
query = """
    SELECT id, regionName, dataDate, MinT, MaxT
    FROM TemperatureForecasts
    WHERE regionName = ?
    ORDER BY dataDate
"""
df_weather_from_db = pd.read_sql_query(query, conn, params=(selected_region,))

print(df_weather_from_db)

conn.close()

   id regionName    dataDate  MinT  MaxT
0   8       中部地區  2026-09-16    22    29
1   9       中部地區  2026-09-17    22    32
2  10       中部地區  2026-09-18    22    33
3  11       中部地區  2026-09-19    23    33
4  12       中部地區  2026-09-20    23    32
5  13       中部地區  2026-09-21    21    32
6  14       中部地區  2026-09-22    21    32


順便輸出一份 CSV，方便檢查或交作業附上：

In [14]:
conn = sqlite3.connect(DB_NAME)
df_all = pd.read_sql_query(
    "SELECT regionName, dataDate, MinT, MaxT FROM TemperatureForecasts ORDER BY id", conn
)
conn.close()

df_all.to_csv("weather_data.csv", index=False, encoding="utf-8-sig")
print(f"已輸出 weather_data.csv（{len(df_all)} 筆）")
df_all.head(10)

已輸出 weather_data.csv（42 筆）


,regionName,dataDate,MinT,MaxT
0,北部地區,2026-09-16,23,27
1,北部地區,2026-09-17,23,30
2,北部地區,2026-09-18,24,31
3,北部地區,2026-09-19,24,31
4,北部地區,2026-09-20,24,30
5,北部地區,2026-09-21,22,30
6,北部地區,2026-09-22,22,30
7,中部地區,2026-09-16,22,29
8,中部地區,2026-09-17,22,32
9,中部地區,2026-09-18,22,33


## HW4-4: Implement the Temperature Forecast Web App (Streamlit)
- 用 Streamlit 建立 Web App
- 提供下拉選單讓使用者選擇地區
- **必須從 SQLite 資料庫用 SQL 查詢資料**
- 用折線圖與表格顯示一週的氣溫資料

### Step 1: 建立 Streamlit app

In [15]:
%%writefile app.py


"""HW4-4：氣溫預報 Web App（Streamlit）

執行方式：
    streamlit run app.py

重點：畫面上所有數字都是「即時用 SQL 從 data.db 查出來的」，
不是從記憶體或 CSV 讀的。
"""

import os
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import streamlit as st

DB_PATH = Path(__file__).parent / "data.db"

# 六大區域的顯示順序（和氣象署的排法一致）
REGION_ORDER = ["北部地區", "中部地區", "南部地區", "東北部地區", "東部地區", "東南部地區"]


# --------------------------------------------------------------------------
# 資料庫查詢：每個函式都對應一句 SQL
# --------------------------------------------------------------------------
def query(sql: str, params: tuple = ()) -> pd.DataFrame:
    """對 data.db 執行一句 SQL，回傳 DataFrame。"""
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)


def get_region_names() -> list:
    """列出資料庫裡所有地區名稱。"""
    sql = "SELECT DISTINCT regionName FROM TemperatureForecasts"
    names = query(sql)["regionName"].tolist()
    # 依照固定順序排列，沒在清單裡的就排在後面
    return sorted(names, key=lambda n: REGION_ORDER.index(n) if n in REGION_ORDER else 99)


def get_forecast(region_name: str) -> pd.DataFrame:
    """查出某個地區一週的最高 / 最低氣溫。

    使用參數化查詢（?）而不是字串拼接，避免 SQL injection。
    """
    sql = """
        SELECT dataDate, MinT, MaxT
        FROM TemperatureForecasts
        WHERE regionName = ?
        ORDER BY dataDate
    """
    return query(sql, (region_name,))


# --------------------------------------------------------------------------
# 畫面
# --------------------------------------------------------------------------
st.set_page_config(page_title="氣溫預報 Web App", page_icon="🌡️", layout="wide")
st.title("🌡️ 氣溫預報 Web App")
st.caption("資料來源：中央氣象署開放資料平臺　|　資料儲存：SQLite (data.db)")

if not DB_PATH.exists():
    st.error("找不到 data.db，請先執行 hw4.ipynb 建立資料庫。")
    st.stop()

# ---------------------------------------------------------------------------
# 側邊欄：重新抓取最新預報
# 需要 CWA 授權碼。本機看環境變數 CWA_API_KEY；
# 部署到 Streamlit Cloud 時改在 Settings → Secrets 設定同名的 secret。
# ---------------------------------------------------------------------------
def get_api_key() -> str:
    try:
        if "CWA_API_KEY" in st.secrets:
            return str(st.secrets["CWA_API_KEY"])
    except Exception:
        pass  # 沒有 secrets 檔案時 st.secrets 會拋例外，忽略即可
    return os.getenv("CWA_API_KEY", "")


with st.sidebar:
    st.header("資料")
    updated_at = (
        datetime.fromtimestamp(DB_PATH.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
        if DB_PATH.exists() else "—"
    )
    st.caption(f"資料庫更新時間：{updated_at}")

    api_key = get_api_key()
    if api_key:
        if st.button("🔄 重新抓取最新預報", use_container_width=True):
            try:
                import weather
                with st.spinner("正在向中央氣象署取得最新預報…"):
                    df_new = weather.refresh_data(api_key)
                st.success(f"已更新 {len(df_new)} 筆資料")
                st.rerun()
            except Exception as error:
                st.error(f"更新失敗：{error}")
    else:
        st.caption(
            "未設定 CWA 授權碼，無法在線上更新。\n\n"
            "本機：`export CWA_API_KEY=\"...\"`\n\n"
            "Streamlit Cloud：Settings → Secrets 新增 `CWA_API_KEY`"
        )


# --- 下拉選單：選擇地區 ---
regions = get_region_names()
selected_region = st.selectbox("選擇地區", regions)

df = get_forecast(selected_region)

if df.empty:
    st.warning(f"資料庫中沒有 {selected_region} 的資料。")
    st.stop()

# --- 摘要數字 ---
col1, col2, col3 = st.columns(3)
col1.metric("一週最高溫", f"{df['MaxT'].max()} °C")
col2.metric("一週最低溫", f"{df['MinT'].min()} °C")
col3.metric("預報天數", f"{len(df)} 天")

# --- 折線圖 ---
st.subheader(f"{selected_region}　一週氣溫趨勢")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df["dataDate"], y=df["MaxT"], name="最高溫 MaxT",
        mode="lines+markers+text", line=dict(color="#e53e3e", width=3),
        text=df["MaxT"], textposition="top center",
    )
)
fig.add_trace(
    go.Scatter(
        x=df["dataDate"], y=df["MinT"], name="最低溫 MinT",
        mode="lines+markers+text", line=dict(color="#3182ce", width=3),
        text=df["MinT"], textposition="bottom center",
        fill="tonexty", fillcolor="rgba(120,160,220,0.15)",
    )
)
fig.update_layout(
    xaxis_title="日期", yaxis_title="氣溫 (°C)",
    hovermode="x unified", height=430,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    margin=dict(t=40, b=40),
)
st.plotly_chart(fig, use_container_width=True)

# --- 表格 ---
st.subheader(f"{selected_region}　一週氣溫資料")
table = df.rename(columns={"dataDate": "日期", "MinT": "最低溫 (°C)", "MaxT": "最高溫 (°C)"})
st.dataframe(table, use_container_width=True, hide_index=True)

# --- 讓助教看到確實是用 SQL 查的 ---
with st.expander("查看本頁使用的 SQL"):
    st.code(
        "-- 下拉選單的地區清單\n"
        "SELECT DISTINCT regionName FROM TemperatureForecasts;\n\n"
        "-- 選定地區的一週氣溫\n"
        "SELECT dataDate, MinT, MaxT\n"
        "FROM TemperatureForecasts\n"
        f"WHERE regionName = '{selected_region}'\n"
        "ORDER BY dataDate;",
        language="sql",
    )

Overwriting app.py


### Step 2: 執行 Streamlit app

在**終端機**執行下面這行，瀏覽器會自動打開 <http://localhost:8501>：

```bash
streamlit run app.py
```

如果是在 Google Colab 上跑，改用下面這個 cell（透過 localtunnel 對外開埠）。

In [16]:
# --- 本機執行（把註解拿掉即可，Ctrl+C 可停止）---
# !streamlit run app.py

# --- Google Colab 執行 ---
# !echo "Tunnel Password: $(curl -s ifconfig.io)"
# !streamlit run app.py &>logs.txt & \
#   npx -y localtunnel --port 8501

---

## 完成檢核

| 題目 | 要求 | 對應位置 |
|------|------|----------|
| HW4-1 | 用 Requests 調用 CWA API 取得一週預報 | `fetch_dataset()` |
| HW4-1 | 用 `json.dumps` 觀察資料 | HW4-1 Step 4 |
| HW4-2 | 找出並提取 MaxT / MinT | `extract_from_county_forecast()` |
| HW4-2 | 用 `json.dumps` 觀察提取的資料 | HW4-2 Step 2 |
| HW4-3 | 建立 `data.db` 與 `TemperatureForecasts` 資料表 | HW4-3 Step 1 |
| HW4-3 | 列出所有地區名稱 | HW4-3 查詢 1 |
| HW4-3 | 列出中部地區的氣溫資料 | HW4-3 查詢 2 |
| HW4-4 | 下拉選單 | `app.py` 的 `st.selectbox` |
| HW4-4 | 折線圖與表格 | `app.py` 的 `st.plotly_chart` / `st.dataframe` |
| HW4-4 | 從 SQLite 用 SQL 查詢 | `app.py` 的 `query()` |